In [3]:
import pandas as pd
import numpy as np
import re

df_raw = pd.read_csv('player_absence_periods_2006_2018.csv') 

df_raw['Player_Raw'] = df_raw['Relinquished'].fillna(df_raw['Acquired'])

# Function to clean player names (remove parentheticals like "(b. 1984)")
def clean_name(name):
    if pd.isna(name): return np.nan
    return str(name).split('/')[0].strip()

df_raw['Player'] = df_raw['Player_Raw'].apply(clean_name)

# Identify the Action (Start vs End)
# If relinquished it means they left, if aquired then they came back
df_raw['Action'] = np.where(df_raw['Relinquished'].notna(), 'Placed_On_IL', 'Activated_From_IL')

# Ensure Date is datetime
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

#Calculate Absence Periods (Start -> End)
df_raw.sort_values(by=['Player', 'Date'], inplace=True)
absence_periods = []

# Group by Player and Team to pair events
for (player, team), group in df_raw.groupby(['Player', 'Team']):
    # Get lists of dates for placements and activations
    placed_dates = group[group['Action'] == 'Placed_On_IL']['Date'].tolist()
    activated_dates = group[group['Action'] == 'Activated_From_IL']['Date'].tolist()
    
    # Pair them up (chronologically)
    # Take the minimum length to ensure we only form complete pairs
    pairs_count = min(len(placed_dates), len(activated_dates))
    
    for i in range(pairs_count):
        start = placed_dates[i]
        end = activated_dates[i]
        
        # Calculate duration
        days_missed = (end - start).days
        
        if days_missed > 0:
            absence_periods.append({
                'Player': player,
                'Team': team,
                'Absence_Start': start,
                'Absence_End': end,
                'Missed_Days': days_missed
            })

#Create the Final Dataframe
df_absences_final = pd.DataFrame(absence_periods)

df_absences_final.to_csv('player_absence_periods_FIXED.csv', index=False)


Loaded 18872 transactions.
Conversion complete!
Created 7701 absence periods.
Columns: ['Player', 'Team', 'Absence_Start', 'Absence_End', 'Missed_Days']


In [ ]:
import pandas as pd
import numpy as np
import re


df_absences = pd.read_csv('player_absence_periods_FIXED.csv')
df_stats = pd.read_csv('player_impact_stats.csv')


# Function to clean player names (same as before to ensure matching)
def clean_player_name(name):
    if pd.isna(name): return np.nan
    name = str(name).split('/')[0].strip()
    name = re.sub(r'\s+\(.*?\)', '', name).strip()
    return name

# Function to convert Season string (e.g., "2006-07") to End Year (e.g., 2007)
def clean_season_year(season_str):
    try:
        # Handles formats like "2006-07" -> 2007
        year_part = str(season_str).split('-')[0]
        return int(year_part) + 1
    except:
        return np.nan

# Apply cleaning
df_stats['Player_Clean'] = df_stats['PLAYER_NAME'].apply(clean_player_name)
df_stats['Season_Year'] = df_stats['SEASON'].apply(clean_season_year)

# Ensure date columns in absence data are datetime objects
df_absences['Absence_Start'] = pd.to_datetime(df_absences['Absence_Start'])
df_absences['Absence_End'] = pd.to_datetime(df_absences['Absence_End'])

# Map absence date to Season Year for NBA
def get_season_from_date(date):
    if date.month >= 10: 
        return date.year + 1
    else:
        return date.year

df_absences['Season_Year'] = df_absences['Absence_Start'].apply(get_season_from_date)


#Calculate Player Weight (The "Impact" Metric) ---
# Weight = Minutes Per Game * Usage Percentage
# This gives high scores to stars and low scores to bench players.

# Avoid division by zero
df_stats['GP'] = df_stats['GP'].replace(0, 1) 

# Calculate Minutes Per Game (MPG) if MIN is total minutes
# If MIN in your data is already 'Minutes Per Game', remove the "/ df_stats['GP']" part.
df_stats['MPG'] = df_stats['MIN'] / df_stats['GP']

# Calculate raw weight
df_stats['Raw_Impact'] = df_stats['MPG'] * df_stats['USG_PCT']

# Create a clean lookup table for merging
df_weight_lookup = df_stats[['Player_Clean', 'Season_Year', 'Raw_Impact']].copy()
# Handle duplicates if a player played for multiple teams in a season (take the average or max impact)
df_weight_lookup = df_weight_lookup.groupby(['Player_Clean', 'Season_Year'])['Raw_Impact'].mean().reset_index()


#Merge Absences with Player Weights ---
df_merged = pd.merge(
    df_absences,
    df_weight_lookup,
    left_on=['Player', 'Season_Year'],
    right_on=['Player_Clean', 'Season_Year'],
    how='left'
)

# Fill missing weights (for players not in the stats file) with a baseline bench value (e.g., 0.05)
# This ensures every absence counts for at least something.
df_merged['Raw_Impact'] = df_merged['Raw_Impact'].fillna(0.05)

daily_impact_scores = {}

for index, row in df_merged.iterrows():
    team = row['Team']
    start_date = row['Absence_Start']
    end_date = row['Absence_End']
    impact = row['Raw_Impact']
    
    # Create a range of dates for this specific absence
    # We constrain the loop to reasonable lengths to avoid infinite loops on bad data
    if pd.notnull(start_date) and pd.notnull(end_date) and end_date >= start_date:
        date_list = pd.date_range(start=start_date, end=end_date)
        
        for date in date_list:
            date_str = date.strftime('%Y-%m-%d')
            key = (date_str, team)
            
            if key not in daily_impact_scores:
                daily_impact_scores[key] = 0.0
            
            # Add this player's impact to the team's total missing impact for that day
            daily_impact_scores[key] += impact

# Convert to DataFrame
impact_df = pd.DataFrame(
    list(daily_impact_scores.items()),
    columns=['Key', 'Total_Missing_Impact']
)
# Split Key back into Date and Team
impact_df[['Date', 'Team']] = pd.DataFrame(impact_df['Key'].tolist(), index=impact_df.index)
impact_df.drop(columns=['Key'], inplace=True)

# Sort and organize
impact_df = impact_df[['Date', 'Team', 'Total_Missing_Impact']].sort_values(by=['Date', 'Team'])


# Save Final Dataset ---
output_file = 'daily_team_injury_impact.csv'
impact_df.to_csv(output_file, index=False)

print("-" * 30)
print("SUCCESS!")
print(f"Daily Impact Scores saved to: {output_file}")
print("Head of the data:")
print(impact_df.head())